In [ ]:
# adopted from https://moscot.readthedocs.io/en/latest/notebooks/tutorials/200_temporal_problem.html

import os
import warnings

import moscot as mt
import moscot.plotting as mpl
from moscot.problems.time import TemporalProblem

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import scanpy as sc

plt.rcParams["legend.scatterpoints"] = 1
sc.set_figure_params(scanpy=True, dpi=80, dpi_save=300)

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [ ]:
# set wd
os.chdir("./analysis/dir/")

In [ ]:
# load adata objects
mono_adata = sc.read_h5ad("seurat_objects/all_samples_Mono_toadata.h5ad")
car_adata = sc.read_h5ad("seurat_objects/CARonly_TNK_toadata.h5ad")
noncar_adata = sc.read_h5ad("seurat_objects/nonCAR_TNK_toadata.h5ad")

In [ ]:
# split non car cells into two 
noncar_adata_1 = noncar_adata[noncar_adata.obs["timepoint"] != "TDN"].copy()

noncar_adata_2 = noncar_adata[~noncar_adata.obs["timepoint"].isin(["APH", "D0"])].copy()

In [ ]:
# timepoint numeric function 
def adapt_time(x):
    if x["timepoint"] == "APH":
        return 1
    if x["timepoint"] == "TDN":
        return 2
    if x["timepoint"] == "D0":
        return 3
    if x["timepoint"] == "Peak":
        return 4
    if x["timepoint"] == "4W":
        return 5
    raise ValueError

In [ ]:
#Pre-processing 
mono_adata.obs["timepoint"] = mono_adata.obs.apply(adapt_time, axis=1).astype("category")
car_adata.obs["timepoint2"] = car_adata.obs.apply(adapt_time, axis=1).astype("category")
noncar_adata.obs["timepoint2"] = noncar_adata.obs.apply(adapt_time, axis=1).astype("category")

sc.pp.highly_variable_genes(mono_adata, n_top_genes=2000, batch_key="sample.name")
sc.pp.highly_variable_genes(car_adata, n_top_genes=2000, batch_key="sample.name")
sc.pp.highly_variable_genes(noncar_adata, n_top_genes=2000, batch_key="sample.name")

noncar_adata_1.obs["timepoint2"] = noncar_adata_1.obs.apply(adapt_time, axis=1).astype("category")
noncar_adata_2.obs["timepoint2"] = noncar_adata_2.obs.apply(adapt_time, axis=1).astype("category")

sc.pp.highly_variable_genes(noncar_adata_1, n_top_genes=2000, batch_key="sample.name")
sc.pp.highly_variable_genes(noncar_adata_2, n_top_genes=2000, batch_key="sample.name")

In [ ]:
# constructing temporal problem
tp0 = TemporalProblem(mono_adata)
tp2 = TemporalProblem(car_adata)
tp4 = TemporalProblem(noncar_adata_1) # Day0 to Peak 
tp5 = TemporalProblem(noncar_adata_2) # TDN to Peak

In [ ]:
# proliferation/apoptosis
tp0 = tp0.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human", use_raw = False 
)

tp2 = tp2.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human", use_raw = False 
)

tp4 = tp4.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human", use_raw = False 
)

tp5 = tp5.score_genes_for_marginals(
    gene_set_proliferation="human", gene_set_apoptosis="human", use_raw = False 
)



In [ ]:
tp0 = tp0.prepare(time_key="timepoint", joint_attr = "X_harmony") # uses sq_euclidean cost function
tp2 = tp2.prepare(time_key="timepoint2", joint_attr = "X_harmony") # uses sq_euclidean cost function
tp4 = tp4.prepare(time_key="timepoint2", joint_attr = "X_harmony") # uses sq_euclidean cost function
tp5 = tp5.prepare(time_key="timepoint2", joint_attr = "X_harmony") # uses sq_euclidean cost function

In [ ]:
tp0 = tp0.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")
tp0.save("./python_objects/moscot_temporalproblem_Monocyte.pickle")

tp2 = tp2.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")
tp2.save("./python_objects/moscot_temporalproblem_CAROnly_TNK.pickle")

tp4 = tp4.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")
tp4.save("./python_objects/moscot_temporalproblem_nonCAR_A-D-P-W_TNK.pickle", overwrite=True)

tp5 = tp5.solve(epsilon=1e-3, tau_a=0.95, scale_cost="mean")
tp5.save("./python_objects/moscot_temporalproblem_nonCAR_T-P-W_TNK.pickle", overwrite=True)